- ### Array

1. [Declaration](#declaration)

2. [Attribution](#attribution)

3. [Operation](#operation)

4. [Search](#search)

5. [Iterator](#iterator)

6. [Function parameters](#function-parameters)


---

- ### Declaration

In [2]:
# Setup for oneline command %%cpp
import os, tempfile, subprocess
from IPython.core.magic import register_cell_magic
import shlex

@register_cell_magic
def cpp(line, cell):
    """
    Usage:
    %%cpp -i "input for cin" -- arg1 arg2 ...
    """
    tokens = shlex.split(line)
    input_data = None
    run_args = []

    # Parse stdin input
    if "-i" in tokens:
        idx = tokens.index("-i")
        if idx + 1 < len(tokens):
            input_data = tokens[idx + 1]

    # Parse program arguments after --
    if "--" in tokens:
        idx = tokens.index("--")
        run_args = tokens[idx + 1:]

    # Write temp C++ file
    with tempfile.NamedTemporaryFile(suffix=".cpp", delete=False, mode="w") as tmp_cpp:
        tmp_cpp.write(cell)
        cpp_path = tmp_cpp.name
    exe_path = cpp_path[:-4] + ".exe"

    try:
        # Compile
        compile_proc = subprocess.run(
            ["g++", "-std=c++23", "-O2", "-Wall", cpp_path, "-o", exe_path],
            capture_output=True,
            text=True
        )
        if compile_proc.returncode != 0:
            print("❌ Compilation failed:\n", compile_proc.stderr)
            return

        # Run program
        run_proc = subprocess.run(
            [exe_path] + run_args,
            input=input_data,      # feed stdin here
            capture_output=True,
            text=True
        )
        if run_proc.stdout:
            print(run_proc.stdout, end="")
        if run_proc.stderr:
            print("⚠️ Runtime error:\n", run_proc.stderr)

    finally:
        for f in (cpp_path, exe_path):
            try: os.remove(f)
            except: pass

**classic c-style array: type [n]**

In [3]:
%%cpp
#include <iostream>
using namespace std;

int main() {
    int a1[3]; // declaration without initialization
    double a2[5] = {1,2,3,4,5}; // initialization all
    float a3[] = {1.0, 2.0, 3.0}; // initialization without size
    for (int i = 0; i < 3; i++) {
        a1[i] = i*i;
        cout << a1[i] << " ";
    }
    cout << endl;
    for (int i = 0; i < sizeof(a2)/sizeof(a2[0]); i++) { // use sizeof to get size
        cout << a2[i] << " ";
    }
    cout << endl;
    for (auto x : a3) { // work only in C++11 and later
        cout << x << " ";
    }
}

0 1 4 
1 2 3 4 5 
1 2 3 

**modern C++ style array: array<type,n>**

In [13]:
%%cpp
#include <iostream>
#include <array>
#include <string>
using namespace std;

int main() {
    array<int, 3> a1; // declaration without initialization, must specify size
    array<string, 3> a2 = {"Hello", "World", "!"}; // initialization

    for (int i = 0; i < a1.size(); i++) {
        a1[i] = i+1;
        cout << a1[i] << " ";
    }
    cout << endl;
    for (auto &x : a2) {
        cout << x << " ";
    }
    cout << endl;
}

1 2 3 
Hello World ! 


---

- ### Attribution

**size**

In [5]:
%%cpp
#include <iostream>
#include <array>
#include <string>
using namespace std;

int main() {
    array<string, 3> arr = {"hello", "world", "!"};
    for (int i = 0; i < arr.size(); i++) {
        arr[i] += " ";
        cout << arr[i] << " ";
    }
}

hello  world  !  

**front and back**

In [ ]:
%%cpp
#include <iostream>
#include <array>
using namespace std;

int main() {
    array<int, 5> arr = {1, 2, 3, 4, 5};
    cout << arr.front() << " " << arr.back() << endl;
}

1 5


**index**

In [ ]:
%%cpp
#include <iostream>
#include <array>
using namespace std;

int main() {
    array<int, 5> arr = {1, 2, 3, 4, 5};
    cout << arr.at(2) << endl; // Access with bounds checking
    cout << arr[2] << endl;    // Access without bounds checking
    cout << arr.data()[2] << endl; // Access via pointer
}

3
3
3


---

- ### Operation

**fill**

In [8]:
%%cpp
#include <iostream>
#include <array>
using namespace std;

int main() {
    array<int, 5> arr = {1, 2, 3, 4, 5};
    arr.fill(10);
    for (int x : arr) {
        cout << x << " ";
    }
}

10 10 10 10 10 

**sort**

In [9]:
%%cpp
#include <iostream>
#include <array>
using namespace std;

int main() {
    array<int, 5> arr = {4, 2, 5, 1, 3};
    sort(arr.begin(), arr.end());
    for (int x : arr) {
        cout << x << " ";
    }
    cout << endl;
    sort(arr.begin(), arr.end(), greater<int>());
    for (int x : arr) {
        cout << x << " ";
    }
    cout << endl;
}


1 2 3 4 5 
5 4 3 2 1 


**reverse**

In [27]:
%%cpp
#include <iostream>
#include <array>
using namespace std;

int main() {
    array<int,5> arr = {1, 2, 3, 4, 5};
    reverse(arr.begin(), arr.end());
    for (int x : arr) {
        cout << x << " ";
    }
}

5 4 3 2 1 

**swap**

In [10]:
%%cpp
#include <iostream>
#include <array>
using namespace std;

int main() {
    array<int, 5> arr = {1, 2, 3, 4, 5};
    array<int, 5> arr2 = {6, 7, 8, 9, 10};
    arr.swap(arr2);
    for (int x : arr) {
        cout << x << " ";
    }
    cout << endl;
    for (int x : arr2) {
        cout << x << " ";
    }
    cout << endl;
}

6 7 8 9 10 
1 2 3 4 5 


**slicing**

In [ ]:
%%cpp
#include <iostream>
#include <array>
#include <algorithm>
using namespace std;

int main() {
    array<int, 5> arr = {1, 2, 3, 4, 5};
    array<int, 3> subarr;
    copy(arr.begin()+2, arr.end(), subarr.begin());
    for (int x : arr) {
        cout << x << " ";
    }
    cout << endl;
    for (int x : subarr) {
        cout << x << " ";
    }
    cout << endl;
}

1 2 3 4 5 
3 4 5 


---

- ### Search

**find**

In [28]:
%%cpp
#include <iostream>
#include <array>
using namespace std;

int main() {
    array<int,5> a = {1, 2, 3, 4, 5};
    auto it = find(a.begin(), a.end(), 3);
    if (it != a.end()) 
    {
        cout << "Found element " << *it << " at index " << distance(a.begin(), it) << endl;
    }
    else 
    {
        cout << "Not found" << endl;
    }
}

Found element 3 at index 2


**count**

In [29]:
%%cpp
#include <iostream>
#include <array>
using namespace std;

int main() {
    array<int,6> a = {1, 2, 3, 4, 5, 3};
    int cnt = count(a.begin(), a.end(), 3);
    cout << "Count of 3: " << cnt << endl;
}

Count of 3: 2


---

- ### Iterator

**pointer**

In [11]:
%%cpp
#include <iostream>
#include <array>
using namespace std;

int main() {
    array<int, 5> arr = {1, 2, 3, 4, 5};
    int* p = arr.data();
    int* q = arr.end()-1;
    for (int i = 0; i < arr.size(); i++) {
        cout << *p << " ";
        ++p;
    }
    cout << endl;
    for (int i = 0; i < arr.size(); i++) {
        cout << *q << " ";
        --q;
    }
    cout << endl;
}

1 2 3 4 5 
5 4 3 2 1 


---

- ### Function parameters

**reference (modifiable)**

In [17]:
%%cpp
#include <iostream>
#include <array>
using namespace std;

void squared(array<int, 5> &arr)
    {
        for (int i = 0; i < arr.size(); i++) 
        {
            arr[i] = arr[i]*arr[i];
            cout << arr[i] << " ";
    }
    }

int main() {
    array<int, 5> a = {1, 2, 3, 4, 5};
    squared(a);
}

1 4 9 16 25 

**const reference (read only)**

In [32]:
%%cpp
#include <iostream>
#include <array>
#include <cmath>
using namespace std;

double norm(const array<double,2> &a)
{
    double sum = 0;
    for (auto x : a)
    {
        sum += pow(x, 2);
    }
    return sqrt(sum);
}

int main() {
    array<double,2> a = {3.0, 4.0};
    cout << "Norm: " << norm(a) << endl;
}

Norm: 5


**value (copied)**

In [31]:
%%cpp
#include <iostream>
#include <array>
using namespace std;

double max(array<double,4> a)
{
    double max_val = a[0];
    for (auto x : a)
    {
        if (x > max_val)
        {
            max_val = x;
        }
    }
    return max_val;
}

int main() {
    array<double,4> a = {1.5, 3.2, 7.4, 2.8};
    cout << "Max: " << max(a) << endl;
}

Max: 7.4
